# 00 — Setup & Raw Landing

**Purpose:** Create the Unity Catalog schemas, then land the uploaded Kaggle *E-Commerce
Customer Behavior* table as a clean raw table. The dataset is small (~350 rows) and contains
real defects — notably missing `Satisfaction Level` values — so the Silver quality pre-checks
catch genuine problems.

### Dataset
- **Kaggle:** [E-Commerce Customer Behavior](https://www.kaggle.com/datasets/uom190346a/e-commerce-customer-behavior-dataset)
  — ~350 rows, one customer per row. Tiny, so it runs instantly on Databricks.
- Uploaded to Unity Catalog as a managed table via **Catalog → Create → Table**.

### What This Notebook Does
1. Creates the catalog schemas (`raw`, `bronze`, `silver`, `gold`).
2. Reads the uploaded source table, **normalises column names (case-insensitive)**, and casts types.
3. Writes `shift_left_optimization.raw.customers_raw` with **automatic Liquid Clustering** (`CLUSTER BY AUTO`).

### Source & Target
| Direction | Table |
|-----------|-------|
| Source (uploaded managed table) | `shift_left_optimization.default.e_commerce_customer_behavior_sheet_1` |
| Target (raw staging) | `shift_left_optimization.raw.customers_raw` |

> **Prerequisites:** the source table already exists (uploaded via the Catalog UI). Update
> `SOURCE_TABLE` below if your catalog/table name differs.

---

In [0]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
CATALOG      = 'shift_left_optimization'
SOURCE_TABLE = f'{CATALOG}.default.e_commerce_customer_behavior_sheet_1'  # uploaded table
RAW_TABLE    = f'{CATALOG}.raw.customers_raw'

for schema in ['raw', 'bronze', 'silver', 'gold']:
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}')

# Idempotency — drop the raw staging table before re-running
spark.sql(f'DROP TABLE IF EXISTS {RAW_TABLE}')

print(f'Source: {SOURCE_TABLE}')
print(f'Target: {RAW_TABLE}')

In [0]:
from pyspark.sql import functions as F

---

### Step 1 — Read the uploaded source table

In [0]:
src_df = spark.read.table(SOURCE_TABLE)

print(f'Source rows: {src_df.count()}')
print(f'Source columns: {src_df.columns}')
display(src_df.limit(5))

---

### Step 2 — Normalise column names and cast types

The upload UI may sanitise headers differently (`Customer ID`, `customer_id`, `Customer_ID`,
...). We lowercase + underscore every column name first, then select a clean, typed schema —
so this works regardless of how the table was created. Missing `satisfaction_level` values are
preserved as `NULL` for the Silver layer to catch.

In [0]:
# Normalise every column name -> lowercase, spaces to underscores (case-insensitive mapping)
renamed_df = src_df
for col_name in src_df.columns:
    clean = col_name.strip().lower().replace(' ', '_')
    renamed_df = renamed_df.withColumnRenamed(col_name, clean)

customers_df = (
    renamed_df
    .select(
        F.col('customer_id').cast('int').alias('customer_id'),
        F.col('gender').alias('gender'),
        F.col('age').cast('int').alias('age'),
        F.col('city').alias('city'),
        F.col('membership_type').alias('membership_type'),
        F.col('total_spend').cast('double').alias('total_spend'),
        F.col('items_purchased').cast('int').alias('items_purchased'),
        F.col('average_rating').cast('double').alias('avg_rating'),
        F.col('discount_applied').cast('boolean').alias('discount_applied'),
        F.col('days_since_last_purchase').cast('int').alias('days_since_last_purchase'),
        F.col('satisfaction_level').alias('satisfaction_level'),
    )
)

print(f'Mapped rows: {customers_df.count()}')

In [0]:
# Land the raw staging table, then enable automatic Liquid Clustering
(customers_df.write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(RAW_TABLE))

spark.sql(f'ALTER TABLE {RAW_TABLE} CLUSTER BY AUTO')   # Databricks picks + evolves keys
print(f'Wrote (CLUSTER BY AUTO): {RAW_TABLE}')

---

### Validation — the defects are real

In [0]:
%sql
SELECT
    COUNT(*)                                                       AS total_rows,
    SUM(CASE WHEN satisfaction_level IS NULL THEN 1 ELSE 0 END)    AS missing_satisfaction,
    SUM(CASE WHEN total_spend <= 0           THEN 1 ELSE 0 END)    AS non_positive_spend,
    SUM(CASE WHEN avg_rating NOT BETWEEN 1 AND 5 THEN 1 ELSE 0 END) AS rating_out_of_range
FROM shift_left_optimization.raw.customers_raw

In [0]:
print(f'Raw row count: {spark.read.table(RAW_TABLE).count()}')
spark.read.table(RAW_TABLE).printSchema()